1. Импорт библиотек и создание SparkSession

In [58]:
# Импорт SparkSession
from pyspark.sql import SparkSession

# Импорт функций PySpark
from pyspark.sql.functions import (
    col, # работа с колонками
    to_timestamp, # преобразование строки в timestamp
    to_date, # извлечение даты
    explode, # развертывание массивов в строки
    sha2, # хеширование (анонимизация данных)
    row_number, desc # row_number (нумерация строк) и desc (сортировка по убыванию)
)

from pyspark.sql.window import Window # для выполнения оконных операций

# Создание SparkSession 
spark = SparkSession.builder \
    .appName("pyspark_json_analysis") \
    .getOrCreate()

2. Загрузка данных

In [59]:
# Чтение JSON файла 
df = spark.read.json("farmers-protest-tweets-2021-03-5.json")

# Чтение CSV файла
# df = spark.read.csv("sales.csv", header=True, inferSchema=True)

# Чтение Parquet файла
# df = spark.read.parquet("sales.parquet")

3. Изучение структуры данных

In [60]:
# Просмотр схемы данных (структура JSON)
df.printSchema()

root
 |-- content: string (nullable = true)
 |-- conversationId: long (nullable = true)
 |-- date: string (nullable = true)
 |-- id: long (nullable = true)
 |-- lang: string (nullable = true)
 |-- likeCount: long (nullable = true)
 |-- media: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- duration: double (nullable = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- previewUrl: string (nullable = true)
 |    |    |-- thumbnailUrl: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- variants: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- bitrate: long (nullable = true)
 |    |    |    |    |-- contentType: string (nullable = true)
 |    |    |    |    |-- url: string (nullable = true)
 |-- mentionedUsers: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- created: string (nullable = true)
 |    |   

In [61]:
# Просмотр первых строк
df.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------+-------------------+-------------------------+-------------------+----+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+--------+----------+-----------+---------------------------------------------------------------------------------------------------------------+----------+------------+--------------+------------------------------------------------------------------------------------+-------------------+-----------------------------------+-----------+--------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

4. Подготовка датасета для анализа

In [62]:
# Формирование упрощенного датасета для анализа
df_simple = df.select(
    "date",
    "content",
    "lang",
    "likeCount",
    "replyCount",
    "retweetCount",
    "user.username",
    "user.followersCount"
)

# Проверка данных
df_simple.show(5, truncate=False)
df_simple.printSchema()

# Преобразование строки даты в timestamp
df_simple = df_simple.withColumn(
    "date",
    to_timestamp("date")
)

+-------------------------+---------------------------------------------------------------------------------------------------------------+----+---------+----------+------------+---------------+--------------+
|date                     |content                                                                                                        |lang|likeCount|replyCount|retweetCount|username       |followersCount|
+-------------------------+---------------------------------------------------------------------------------------------------------------+----+---------+----------+------------+---------------+--------------+
|2021-03-30T03:33:46+00:00|Support 👇\n\n#FarmersProtest                                                                                  |en  |0        |0         |0           |ShashiRajbhar6 |1788          |
|2021-03-30T03:33:23+00:00|Supporting farmers means supporting our country #StopHateAgainstFarmers #FarmersProtest https://t.co/iWPatTK6NV|en  |0        |0      

Метрики анализа

1 метрика: Топ пользователей по количеству твитов

In [63]:
# Анализ активности пользователей
user_activity = (
    df_simple
    .filter(col("username").isNotNull()) # Убираем пустые значения
    .groupBy("username") # Группировка по пользователю
    .count()  
    .withColumnRenamed("count", "num_posts") # Переименование колонки
    .orderBy(col("num_posts").desc()) # Сортировка по убыванию активности
)

# Вывод топ-10 пользователей
user_activity.show(10)

+---------------+---------+
|       username|num_posts|
+---------------+---------+
|harjot_tweeting|     7134|
|  tasveersandhu|     2091|
|shells_n_petals|     1991|
|         jot__b|     1841|
|  rebelpacifist|     1806|
|       rumsomal|     1722|
|    Iamjazzie96|     1502|
|       Jass_k_G|     1460|
|DigitalKisanBot|     1453|
|  z_khalique007|     1446|
+---------------+---------+
only showing top 10 rows


2 метрика: Активность по дням

In [64]:
# Достаем только дату (без времени)
df_time = df_simple.withColumn("day", to_date("date"))

# Сколько твитов в каждый день
daily_activity = (
    df_time
    .groupBy("day")
    .count()
    .orderBy("day")
)

daily_activity.show(5)

+----------+-----+
|       day|count|
+----------+-----+
|2021-02-01| 5096|
|2021-02-02|23021|
|2021-02-03|82966|
|2021-02-04|62616|
|2021-02-05|33616|
+----------+-----+
only showing top 5 rows


3 метрика: Анализ вовлеченности пользователей

In [65]:
activity = (
    df_simple
    .groupBy("username")
    .sum("likeCount", "retweetCount", "replyCount")
    .withColumnRenamed("sum(likeCount)", "likes")
    .withColumnRenamed("sum(retweetCount)", "retweets")
    .withColumnRenamed("sum(replyCount)", "replies")
    .orderBy(col("retweets").desc())
)

activity.show(10)

+--------------+------+--------+-------+
|      username| likes|retweets|replies|
+--------------+------+--------+-------+
|       rihanna|944307|  315547| 163065|
| GretaThunberg|610101|  186824| 100094|
|   RaviSinghKA|435103|  124916|  12465|
|  vanessa_vash|408703|  120844|   9119|
|     amaanbali|300735|   92250|   8175|
|      NavJammu| 60765|   74974|   2591|
|saahilmenghani|206516|   73069|   3596|
|    miakhalifa|239186|   62893|  24651|
|          ndtv|241604|   53565|  11013|
|Jamie_Margolin|122973|   41466|   3970|
+--------------+------+--------+-------+
only showing top 10 rows


4 метрика: Анализ по языкам

In [67]:
lang_activity = (
    df_simple
    .groupBy("lang")
    .count()
    .orderBy(col("count").desc())
)

lang_activity.show()

+----+------+
|lang| count|
+----+------+
|  en|417511|
|  tl|   609|
|  et|   339|
|  in|   218|
|  fr|   152|
|  pt|   113|
|  es|    95|
|  sv|    66|
|  eu|    65|
|  ht|    60|
|  it|    55|
|  de|    52|
|  nl|    40|
|  da|    38|
|  vi|    37|
|  no|    13|
|  ca|    12|
|  tr|     8|
|  pl|     6|
|  hu|     4|
+----+------+
only showing top 20 rows


Работа с персональными данными пользователя

In [68]:
# Извлекаем персональные данные пользователя
user_data = df.select(
    col("user.id").alias("user_id"),
    col("user.username").alias("username"),
    col("user.followersCount").alias("followers_count"),
    col("user.location").alias("location")
)

user_data.show(10, truncate=False)

+-------------------+---------------+---------------+------------------------+
|user_id            |username       |followers_count|location                |
+-------------------+---------------+---------------+------------------------+
|1015969769760096256|ShashiRajbhar6 |1788           |Azm Uttar Pradesh, India|
|1332937272581263362|kaursuk06272818|51             |                        |
|1332937272581263362|kaursuk06272818|51             |                        |
|1308356658582618112|SukhdevSingh_  |2595           |Punjab, India           |
|1357311756532649985|Davidmu66668113|18             |                        |
|2918610912         |Abhimanyu_1987 |173            |Jaipur,Rajasthan,India  |
|1237420463581728768|singhwhotweets |3786           |Panjab                  |
|1361422240223350784|J_cheeselover  |184            |                        |
|1309302936317636608|DigitalKisanBot|2328           |Atlanta, GA             |
|858383035137273856 |rajriarhughes  |643            

Анонимизация данных (хеширование)

In [69]:
# Защита персональных данных (анонимизация username)
user_data_secure = user_data.withColumn(
    "username_hash",
    sha2(col("username"), 256)
)

user_data_secure.show(10, truncate=False)

+-------------------+---------------+---------------+------------------------+----------------------------------------------------------------+
|user_id            |username       |followers_count|location                |username_hash                                                   |
+-------------------+---------------+---------------+------------------------+----------------------------------------------------------------+
|1015969769760096256|ShashiRajbhar6 |1788           |Azm Uttar Pradesh, India|367118eb287e70c5c020c4917b3b93d893c6ae88aa8f1c1efb1b012ffb27a9b1|
|1332937272581263362|kaursuk06272818|51             |                        |264427390f2e3bc9b9c8fb13b52f594196ceff94fd3c792c2d09a839754eff51|
|1332937272581263362|kaursuk06272818|51             |                        |264427390f2e3bc9b9c8fb13b52f594196ceff94fd3c792c2d09a839754eff51|
|1308356658582618112|SukhdevSingh_  |2595           |Punjab, India           |8a893000ec2bfea4fc3551771821d060b87f91699341caaaaefb23381b

Работа с массивами

In [70]:
# Извлечение упоминаний пользователей из массива
mentions_users = (
    df
    .select(explode("mentionedUsers").alias("m"))
    .select(col("m.username").alias("mentioned_username"))
)

mentions_users.show(10)

+------------------+
|mentioned_username|
+------------------+
|     Troll48611422|
|      Doabapunjab1|
|   Terminator00096|
|          HarjasK1|
|   _theshivatribe_|
|   Arshdee00485709|
|        tamankaur_|
|       sukhisukhii|
|   WalterW28442171|
|         amaanbali|
+------------------+
only showing top 10 rows


In [71]:
# Топ упоминаемых пользователей
top_mentions = (
    mentions_users
    .groupBy("mentioned_username")
    .count()
    .orderBy(col("count").desc())
)

top_mentions.show(10)

+------------------+-----+
|mentioned_username|count|
+------------------+-----+
|           rihanna|27512|
|     GretaThunberg|15354|
|      narendramodi| 9630|
|       meenaharris| 8059|
|       KanganaTeam| 7323|
|     UNHumanRights| 6585|
|       akshaykumar| 6369|
|          PMOIndia| 6078|
|   Kisanektamorcha| 5917|
|       AmandaCerny| 5771|
+------------------+-----+
only showing top 10 rows


Использование Spark SQL для анализа данных

In [72]:
# Создание временного представления (view)
df_simple.createOrReplaceTempView("tweets")

# SQL-запрос: анализ активности по языкам
lang_sql = spark.sql("""
    SELECT 
        lang,
        COUNT(*) as tweet_count
    FROM tweets
    GROUP BY lang
    ORDER BY tweet_count DESC
""")

lang_sql.show()

+----+-----------+
|lang|tweet_count|
+----+-----------+
|  en|     417511|
|  tl|        609|
|  et|        339|
|  in|        218|
|  fr|        152|
|  pt|        113|
|  es|         95|
|  sv|         66|
|  eu|         65|
|  ht|         60|
|  it|         55|
|  de|         52|
|  nl|         40|
|  da|         38|
|  vi|         37|
|  no|         13|
|  ca|         12|
|  tr|          8|
|  pl|          6|
|  hu|          4|
+----+-----------+
only showing top 20 rows


Объединение данных

In [73]:
# Первая таблица (основные данные)
table_q = df_simple.select(
    col("username"),
    col("likeCount")
)

# Вторая таблица (дополнительные данные)
table_2 = df_simple.select(
    col("username"),
    col("retweetCount")
)

# JOIN двух таблиц по ключу username
joined_df = table_q.join(
    table_2,        # вторая таблица
    on="username",  # ключ соединения
    how="inner"     # inner join: оставляем только совпадающие username. Другие виды join: left, right, outer
)

# вывод результата
joined_df.show(10)

+---------------+---------+------------+
|       username|likeCount|retweetCount|
+---------------+---------+------------+
| ShashiRajbhar6|        0|           0|
|kaursuk06272818|        0|           0|
|kaursuk06272818|        0|           1|
|kaursuk06272818|        0|           1|
|kaursuk06272818|        0|           1|
|kaursuk06272818|        0|           0|
|kaursuk06272818|        0|           0|
|kaursuk06272818|        0|           1|
|kaursuk06272818|        0|           0|
|kaursuk06272818|        0|           0|
+---------------+---------+------------+
only showing top 10 rows


Оконные функции 

In [74]:
# Окно: сортировка внутри каждого пользователя по количеству лайков
window_spec = Window.partitionBy("username").orderBy(desc("likeCount"))

# Добавляем ранжирование твитов внутри каждого пользователя
df_window = df_simple.withColumn(
    "tweet_rank",
    row_number().over(window_spec)
)

df_window.select("username", "likeCount", "tweet_rank").show(10)

+--------------+---------+----------+
|      username|likeCount|tweet_rank|
+--------------+---------+----------+
|     0001Sidhu|        0|         1|
|    0007_gopal|        0|         1|
|        0011JB|        0|         1|
|     001Sardar|       29|         1|
|005ankitchopra|        1|         1|
|     007anunay|        0|         1|
|007vishalYadav|        3|         1|
|007vishalYadav|        2|         2|
|    009Sandman|        0|         1|
|        011O11|       77|         1|
+--------------+---------+----------+
only showing top 10 rows
